# CUDA Lab 02: Advanced Geometries & Stencils

Nuradil Abyz · Student ID **230103188** · 8 October 2026

Select **Runtime → Change runtime type → T4 GPU**, then run all cells. The full-sized tasks, numerical checks, measured timings and official token are generated during execution.


In [ ]:
!nvidia-smi


In [ ]:
%pip -q install "numba-cuda[cu12]"


## Source files
Each cell below writes a standalone module required by the assignment.


In [ ]:
%%writefile task1_divergence.py
"""Uniform, interleaved and warp-aligned arithmetic; 10 warmed-up trials."""
import json
import time
import numpy as np
from numba import cuda, float32

@cuda.jit
def uniform_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.size:
        value = d_in[idx]
        for _ in range(1000):
            value = value * float32(1.0001) + float32(0.0001)
        d_out[idx] = value

@cuda.jit
def interleaved_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.size:
        value = d_in[idx]
        if idx % 2 == 0:
            for _ in range(1000):
                value = value * float32(1.0001) + float32(0.0001)
        else:
            for _ in range(1000):
                value = (value - float32(0.0001)) / float32(1.0001)
        d_out[idx] = value

@cuda.jit
def warp_aligned_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.size:
        value = d_in[idx]
        warp_id = idx // 32
        if warp_id % 2 == 0:
            for _ in range(1000):
                value = value * float32(1.0001) + float32(0.0001)
        else:
            for _ in range(1000):
                value = (value - float32(0.0001)) / float32(1.0001)
        d_out[idx] = value

def benchmark():
    N = 2**20
    # Separate output keeps all warm-ups and trials on identical input.
    d_in = cuda.to_device(np.ones(N, dtype=np.float32))
    d_out = cuda.device_array(N, dtype=np.float32)
    result = {}
    plus = minus = np.float32(1)
    for _ in range(1000):
        plus = plus*np.float32(1.0001) + np.float32(0.0001)
        minus = (minus-np.float32(0.0001))/np.float32(1.0001)
    idx = np.arange(N)
    for name, kernel in [('A_uniform', uniform_kernel), ('B_interleaved', interleaved_kernel), ('C_warp_aligned', warp_aligned_kernel)]:
        kernel[(N+255)//256, 256](d_in, d_out)
        cuda.synchronize()
        samples = []
        for _ in range(10):
            cuda.synchronize()
            start = time.perf_counter()
            kernel[(N+255)//256, 256](d_in, d_out)
            cuda.synchronize()
            samples.append((time.perf_counter()-start)*1000)
        out = d_out.copy_to_host()
        expected = np.full(N, plus, dtype=np.float32) if name.startswith('A') else np.where((idx if name.startswith('B') else idx//32) % 2 == 0, plus, minus)
        # Fused multiply-add may differ slightly from NumPy's separate operations.
        assert np.allclose(out, expected, rtol=2e-4, atol=1e-4), name
        result[name] = {'mean_ms': float(np.mean(samples)), 'trials_ms': samples, 'max_delta': float(np.max(np.abs(out-expected)))}
        print(f'{name}: {np.mean(samples):.6f} ms (mean of 10 trials)')
    return result

if __name__ == '__main__':
    from pathlib import Path
    Path('benchmark_results.json').write_text(json.dumps(benchmark(), indent=2)+'\n')


In [ ]:
%%writefile task2_stencil_1d.py
"""Three-point smoothing with replicated boundary values."""
import numpy as np
from numba import cuda

@cuda.jit
def stencil_1d(d_in, d_out, N):
    idx = cuda.grid(1)
    if idx < N:
        left = d_in[0] if idx == 0 else d_in[idx - 1]
        right = d_in[N - 1] if idx == N - 1 else d_in[idx + 1]
        d_out[idx] = 0.25 * left + 0.5 * d_in[idx] + 0.25 * right

def run_stencil(h_in):
    arr = np.ascontiguousarray(h_in, dtype=np.float32)
    if arr.ndim != 1:
        raise ValueError('Expected a 1D array')
    if not arr.size:
        return arr.copy()
    d_in = cuda.to_device(arr)
    d_out = cuda.device_array_like(arr)
    stencil_1d[(arr.size + 255) // 256, 256](d_in, d_out, arr.size)
    return d_out.copy_to_host()

def cpu_stencil(arr):
    padded = np.pad(arr, (1, 1), mode='edge')
    return 0.25 * padded[:-2] + 0.5 * padded[1:-1] + 0.25 * padded[2:]

if __name__ == '__main__':
    arr = np.sin(np.linspace(0, 10, 100007)).astype(np.float32)
    out, ref = run_stencil(arr), cpu_stencil(arr)
    assert np.allclose(out, ref, atol=1e-4)
    print(f'TASK 2 PASSED: MAX DELTA = {np.max(np.abs(out-ref))}')


In [ ]:
%%writefile task3_grid_stride.py
"""Scale arbitrary-size vectors using exactly 64 x 256 threads."""
import numpy as np
from numba import cuda

@cuda.jit
def grid_stride_scale_kernel(d_arr, factor, N):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, N, stride):
        d_arr[i] = d_arr[i] * factor

def run_grid_stride(h_arr, factor):
    arr = np.ascontiguousarray(h_arr, dtype=np.float32)
    if arr.ndim != 1:
        raise ValueError('Expected a 1D array')
    if not arr.size:
        return arr.copy()
    d_arr = cuda.to_device(arr)
    threads_per_block = 256
    blocks_per_grid = 64
    grid_stride_scale_kernel[blocks_per_grid, threads_per_block](d_arr, np.float32(factor), arr.size)
    return d_arr.copy_to_host()

if __name__ == '__main__':
    N, factor = 2**24, 4.25
    result = run_grid_stride(np.ones(N, dtype=np.float32), factor)
    assert np.allclose(result, factor)
    print(f'TASK 3 PASSED: {N} elements equal {factor}; 64 x 256 = 16384 threads')


In [ ]:
%%writefile task4_sobel_2d.py
"""Sobel-X correlation with zero output on all four image borders."""
import numpy as np
from numba import cuda

@cuda.jit
def sobel_x_kernel(d_in, d_out, rows, cols):
    col, row = cuda.grid(2)
    if row < rows and col < cols:
        if 0 < row < rows - 1 and 0 < col < cols - 1:
            d_out[row, col] = (
                -1.0 * d_in[row-1, col-1] + 1.0 * d_in[row-1, col+1]
                -2.0 * d_in[row, col-1] + 2.0 * d_in[row, col+1]
                -1.0 * d_in[row+1, col-1] + 1.0 * d_in[row+1, col+1])
        else:
            d_out[row, col] = 0.0

def run_sobel(h_img):
    img = np.ascontiguousarray(h_img, dtype=np.float32)
    if img.ndim != 2:
        raise ValueError('Expected a 2D image')
    if not img.size:
        return img.copy()
    rows, cols = img.shape
    d_in = cuda.to_device(img)
    d_out = cuda.device_array_like(img)
    threads_2d = (16, 16)
    grid = ((cols + 15) // 16, (rows + 15) // 16)
    sobel_x_kernel[grid, threads_2d](d_in, d_out, rows, cols)
    return d_out.copy_to_host()

def cpu_sobel(img):
    out = np.zeros_like(img)
    out[1:-1, 1:-1] = (-img[:-2, :-2] + img[:-2, 2:]
        - 2*img[1:-1, :-2] + 2*img[1:-1, 2:]
        - img[2:, :-2] + img[2:, 2:])
    return out

if __name__ == '__main__':
    img = np.random.default_rng(230103188).random((2048, 2048), dtype=np.float32)
    out, ref = run_sobel(img), cpu_sobel(img)
    assert np.allclose(out, ref, atol=1e-4)
    assert np.all(out[[0, -1], :] == 0) and np.all(out[:, [0, -1]] == 0)
    print(f'TASK 4 PASSED: shape={img.shape}; MAX DELTA = {np.max(np.abs(out-ref))}')


In [ ]:
%%writefile verify_submission.py
"""Instructor verification suite; only PDF wrapping/indentation repaired."""
import sys
import hashlib
import numpy as np
from numba import cuda

def main():
    print("=" * 60)
    print("RUNNING CUDA LAB 02 AUTONOMOUS VERIFICATION")
    print("=" * 60)

    student_id = input("Enter your Student ID: ").strip()
    if not student_id:
        print("[FAIL] Student ID cannot be empty.")
        sys.exit(1)

    # 1. Verify Task 2
    try:
        import task2_stencil_1d as t2
        N = 10007
        test_in = np.sin(np.linspace(0, 10, N)).astype(np.float32)
        h_gpu = t2.run_stencil(test_in)
        h_cpu = t2.cpu_stencil(test_in)
        assert np.allclose(h_gpu, h_cpu, atol=1e-4), "Task 2 output mismatch against CPU"
        print("[PASS] Task 2 (1D Stencil & Clamping)")
    except Exception as e:
        print(f"[FAIL] Task 2: {e}")
        sys.exit(1)

    # 2. Verify Task 3
    try:
        import task3_grid_stride as t3
        N = 100000
        test_arr = np.ones(N, dtype=np.float32)
        factor = 4.25
        res = t3.run_grid_stride(test_arr, factor)
        assert np.allclose(res, factor), "Task 3 elements not uniformly scaled"
        print("[PASS] Task 3 (Grid-Stride Scaling)")
    except Exception as e:
        print(f"[FAIL] Task 3: {e}")
        sys.exit(1)

    # 3. Verify Task 4
    try:
        import task4_sobel_2d as t4
        test_img = np.ones((64, 64), dtype=np.float32)
        sobel_res = t4.run_sobel(test_img)
        # Uniform image must have zero interior gradient
        assert np.max(np.abs(sobel_res[1:-1, 1:-1])) < 1e-5, "Task 4 interior gradient != 0 for flat field"
        assert np.all(sobel_res[0, :] == 0.0), "Task 4 border rows not zeroed"
        assert np.all(sobel_res[:, 0] == 0.0), "Task 4 border columns not zeroed"
        print("[PASS] Task 4 (2D Sobel Horizontal)")
    except Exception as e:
        print(f"[FAIL] Task 4: {e}")
        sys.exit(1)

    # Cryptographic Checksum Token Generation
    hasher = hashlib.sha256()
    hasher.update(student_id.encode('utf-8'))
    try:
        device_name = cuda.get_current_device().name
        hasher.update(device_name if isinstance(device_name, bytes) else device_name.encode('utf-8'))
    except Exception:
        hasher.update(b"UNKNOWN_CUDA_DEVICE")
    hasher.update(h_gpu[:32].tobytes())
    hasher.update(sobel_res[:8, :8].tobytes())
    token = hasher.hexdigest()[:20].upper()

    print("\n" + "=" * 60)
    print(f"VERIFICATION SUCCESSFUL")
    print(f"OFFICIAL SUBMISSION TOKEN: {token}")
    print("=" * 60)
    print("Copy this token directly into your README.md.\n")

if __name__ == "__main__":
    main()





In [ ]:
%%writefile run_lab.py
"""Execute every required size and generate the report from measured results."""
from pathlib import Path
import contextlib, io, json, re, subprocess, sys
from datetime import datetime, timezone
import numpy as np
from numba import cuda
import task1_divergence as t1
import task2_stencil_1d as t2
import task3_grid_stride as t3
import task4_sobel_2d as t4

STUDENT_ID = "230103188"
assert cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
assert not cuda.config.ENABLE_CUDASIM, "Real CUDA GPU required"
device = cuda.get_current_device()
name = device.name.decode() if isinstance(device.name, bytes) else device.name
cc = ".".join(map(str, device.compute_capability))
telemetry = subprocess.check_output(["nvidia-smi"], text=True)
print(telemetry)
Path("nvidia-smi.txt").write_text(telemetry)
print(f"GPU: {name}; compute capability: {cc}")
bench = t1.benchmark()
Path("benchmark_results.json").write_text(json.dumps(bench, indent=2)+"\n")
checks = []
for task in ["task2_stencil_1d.py", "task3_grid_stride.py", "task4_sobel_2d.py"]:
    result = subprocess.run([sys.executable, task], capture_output=True, text=True)
    print(result.stdout, end="")
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    checks.append(result.stdout.strip())
# Additional cases verify halos, non-multiple grids, arbitrary factors and Sobel sign.
for n in (1, 2, 255, 257, 10007):
    a = np.random.default_rng(n).random(n, dtype=np.float32)
    assert np.allclose(t2.run_stencil(a), t2.cpu_stencil(a), atol=1e-4)
for n in (1, 16383, 16385, 100000):
    a = np.linspace(-2, 3, n, dtype=np.float32)
    assert np.allclose(t3.run_grid_stride(a, -1.75), a*np.float32(-1.75), atol=1e-5)
for shape in ((1, 1), (2, 7), (17, 31), (33, 19)):
    img = np.random.default_rng(7).random(shape, dtype=np.float32)
    assert np.allclose(t4.run_sobel(img), t4.cpu_sobel(img), atol=1e-4)
ramp = np.tile(np.arange(31, dtype=np.float32), (17, 1))
assert np.all(t4.run_sobel(ramp)[1:-1, 1:-1] == 8)
print("ADDITIONAL BOUNDARY AND NONUNIFORM-INPUT CHECKS PASSED")
verified = subprocess.run([sys.executable, "verify_submission.py"], input=STUDENT_ID+"\n", capture_output=True, text=True)
print(verified.stdout)
assert verified.returncode == 0, verified.stderr
Path("verification_output.txt").write_text(verified.stdout)
token = re.search(r"OFFICIAL SUBMISSION TOKEN: ([A-F0-9]{20})", verified.stdout).group(1)
rows = "\n".join(f"| {key} | {val['mean_ms']:.6f} |" for key, val in bench.items())
ratio = bench["B_interleaved"]["mean_ms"]/bench["C_warp_aligned"]["mean_ms"]
readme = f"""# CUDA Lab 02: Advanced Geometries & Stencils

**Student ID:** {STUDENT_ID}  
**Allocated GPU Node:** {name}  
**CUDA Compute Capability:** {cc}  
**Official Verification Token:** {token}

## Execution

Executed on Google Colab, UTC: {datetime.now(timezone.utc).isoformat()}.
The source assignment is `CUDA Python Lab 01.pdf` (its internal title is Lab 02).
Files are placed in the requested `Paralel/week-6/08.10.26 - week - 6 - practice_task` folder.
The PDF separately specifies a repository named `cuda-lab-02-230103188`; this submission uses the folder requested by the student.

Open `CUDA_Lab02_230103188.ipynb` in Colab, choose **Runtime > Change runtime type > T4 GPU**, then run all cells.
The notebook writes the Python modules, runs the full-size tasks and instructor verifier, and regenerates this report.
For standalone execution in a CUDA environment: `python run_lab.py`.
For the instructor check alone: `python verify_submission.py`, then enter `{STUDENT_ID}`.

## Task 1: Warp divergence benchmark

N = 1,048,576 float32 values; 1,000 iterations per value; 256 threads per block.
Each kernel has one untimed warm-up and 10 timed trials. Timers use `time.perf_counter()` with `cuda.synchronize()`.
Device buffers are allocated before timing; transfers and validation occur outside timing.
Wall-clock measurements include launch and synchronization overhead, as specified by the assignment.
Every trial reads the same input and writes a separate output.

| Kernel | Mean time (ms), 10 trials |
| --- | ---: |
{rows}

Interleaved / warp-aligned time ratio: **{ratio:.3f}x**.
A uses `v = v * 1.0001 + 0.0001`. B and C use that same path or `v = (v - 0.0001) / 1.0001`.
B assigns opposite paths to adjacent lanes; C assigns one path to each complete 32-thread warp.
B and C process equal numbers of values on each path, so their comparison isolates branch arrangement more directly than A versus B.
Divergence can serialize paths, but the measured slowdown also depends on compiler transformations, arithmetic cost and scheduling; an exact 2x penalty is not guaranteed.
All three outputs were checked against CPU arithmetic; small fused-operation rounding differences are allowed.
Raw trial times and maximum validation errors are in `benchmark_results.json`.

## Tasks 2-4: Results

{chr(10).join('- '+c for c in checks)}

- Stencil: replicated left/right boundary, 256 threads per block, CPU reference uses edge padding.
- Grid-stride: exactly 64 blocks x 256 threads = 16,384 threads; each thread processes 1,024 values at N = 2**24.
- Sobel-X: 16x16 threads per block; dynamic grid is 128x128 blocks for 2048x2048 input. All four output borders are zero.
- Sobel follows the signed formula in the PDF (cross-correlation convention), with a positive derivative along increasing columns.
- Extra GPU checks passed for singleton/two-element arrays, odd dimensions, partial blocks, negative scaling and a column ramp with expected gradient +8.

## Official verification

The provided verifier is preserved, with PDF line wrapping and indentation repaired.
Its smaller smoke-test sizes are run in addition to the full-size task requirements.

```text
{verified.stdout.strip()}
```

## Environment

See `nvidia-smi.txt` and `environment.txt` for runtime telemetry and package versions.
Numba-CUDA installation reference: https://nvidia.github.io/numba-cuda/user/installation.html
"""
Path("README.md").write_text(readme)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
Path("environment.txt").write_text(versions)
print("README.md generated from real GPU results.")


## Run all experiments and the official verifier
Includes full-size tasks and additional boundary tests. Timings are collected only after JIT warm-up.


In [ ]:
%run run_lab.py


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
display(Markdown(Path('README.md').read_text()))


## Download submission files
The archive includes the report, modules, raw measurements and verification output. Save/download this executed notebook separately to preserve its cell outputs.


In [ ]:
from zipfile import ZipFile
from google.colab import files
submission_files = ['README.md', 'task1_divergence.py', 'task2_stencil_1d.py', 'task3_grid_stride.py', 'task4_sobel_2d.py', 'verify_submission.py', 'run_lab.py', 'benchmark_results.json', 'verification_output.txt', 'nvidia-smi.txt', 'environment.txt']
with ZipFile('cuda-lab-02-230103188.zip', 'w') as archive:
    for name in submission_files:
        archive.write(name)
files.download('cuda-lab-02-230103188.zip')
